# 01 · Sentinel-2 data inventory, Novi Sad 2021

Goal: confirm what imagery exists for the 2021 season, check the reflectance offset, and look at one clear July scene on the UTM 34N grid.

Runs in Colab or locally. Only metadata plus one downsampled scene is downloaded.

In [ ]:
# Colab only: uncomment
# !pip install -q git+https://github.com/<your-username>/agro-sat.git

In [ ]:
from agrosat import config
from agrosat.data import (search_items, items_to_frame, scenes_per_month,
                          clearest_item, load_cube, to_reflectance, rgb_stretch)

items = list(search_items())
frame = items_to_frame(items)
print(f"{len(items)} items, {frame['date'].nunique()} unique dates")
frame.head()

## Scenes per month and MGRS tile

A month with 0–1 unique dates will give a weak median composite. Note it now and plan to merge it with a neighbouring month on Day 3.

In [ ]:
scenes_per_month(frame)

## Reflectance offset check

All 2021 items should be baseline < 04.00, so `needs_offset_fix` must be 0. For 2025/2026 data this changes; see the README.

In [ ]:
print('Baselines:', sorted(frame['baseline'].dropna().unique()))
print('Items needing offset fix:', int(frame['needs_offset_fix'].sum()))
print('Earth Search flag present:', 'earthsearch:boa_offset_applied' in items[0].properties)

## Clearest July scene, true colour

Loaded at 20 m to keep the download small. Day 3 loads the full 10 m cube.

In [ ]:
import matplotlib.pyplot as plt

july = clearest_item(items, month=7)
same_day = [it for it in items if it.datetime.date() == july.datetime.date()]
cube = load_cube(same_day, bands=('red', 'green', 'blue'), resolution=20)
flags = items_to_frame(same_day).set_index('date')['needs_offset_fix']
refl = to_reflectance(cube, flags, bands=('red', 'green', 'blue')).isel(time=0).compute()

print(refl.odc.geobox)  # CRS, resolution and shape of the grid
plt.figure(figsize=(10, 8))
plt.imshow(rgb_stretch(refl.red.values, refl.green.values, refl.blue.values))
plt.title(f'Sentinel-2 true colour {july.datetime:%Y-%m-%d}')
plt.axis('off');

## Sanity checks

- Reflectance of vegetation in red is about 0.02–0.08 and in NIR about 0.3–0.5.
- The Danube and Novi Sad should be clearly visible; field patterns should not look shifted or blurred.

In [ ]:
refl[['red', 'green', 'blue']].quantile([0.02, 0.5, 0.98]).to_dataframe()